# Lab 3b: the same four stages, with real models

**Time: about 20 minutes, of which the run itself is three to five. Work in your Lab 3 pairs.**

**Do Lab 3 first.** In Lab 3 you measured four stages of search on 20 judged questions, using free
stand-ins that run with no keys. Two of the stages barely moved the score, and you were told to report
what you measured. This notebook asks the obvious next question: were those stages weak, or were the
stand-ins weak? It reruns the same four stages on the same 20 questions, with the stand-ins swapped for
models this Databricks workspace already serves.

| Part | Lab 3 | This notebook |
|---|---|---|
| Meaning search | small offline model | `databricks-gte-large-en`, an embedding model (it turns text into a list of numbers) |
| Keyword search and merging | the lab's own code | the lab's own code, unchanged |
| Reranker | stand-in built from word counts | `databricks-meta-llama-3-3-70b-instruct`, a chat model that grades each passage 0 to 3 |
| Scanned PDF | not read | read with `ai_parse_document`, Databricks' own document reader |

### Before you start

1. **Your workspace needs Databricks' built-in models.** The notebook uses the pay-per-use models
   `databricks-gte-large-en` and `databricks-meta-llama-3-3-70b-instruct`, and `ai_parse_document`.
   Most workspaces with serverless compute have them; a free edition workspace may limit them.
2. **Nothing to set up.** No keys: the notebook signs in as you. A run takes three to five minutes,
   costs a few cents, billed to your workspace, and leaves nothing running afterwards.
3. **If the models refuse you,** the notebook says so in Step 1 and shows the recorded run from
   8 October instead. You can still do the write-up from that.

Click **Run all**, then read each step while it runs.

### Setup

The next two cells are the same as in every lab notebook. The first installs the readers for PDF,
Word, PowerPoint and Excel files and the keyword search library. The second copies the lab code and the seven documents from this
repository into this session's own folder, so they are the ones you used in Lab 1 and Lab 3.

**What you should see:** a line ending `corpus files: 7`.

In [ ]:
%pip install -q pypdf python-docx python-pptx openpyxl rank_bm25

In [ ]:
# Setup for Databricks. Run this cell first: every later cell depends on it.
# It puts the lab files (src/ and data/) in this session's own folder, /tmp/freshcart, and puts
# src/ on the import path so `from freshcart import ...` works.
# If you opened this notebook from a Git folder of the repository, the files are copied from that
# folder. If you imported only this notebook, they are downloaded from GitHub.
import io, os, sys, pathlib, shutil, textwrap, urllib.request, zipfile

REPO_ZIP = "https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip"
ROOT = pathlib.Path("/tmp/freshcart")


def repo_folder():
    here = pathlib.Path(os.getcwd())          # on Databricks, the folder that holds this notebook
    for p in (here, *here.parents):
        if (p / "src" / "freshcart.py").is_file() and (p / "data" / "raw").is_dir():
            return p
    return None


ROOT.mkdir(parents=True, exist_ok=True)
found = repo_folder()
if found:
    for part in ("src", "data"):
        shutil.copytree(found / part, ROOT / part, dirs_exist_ok=True, copy_function=shutil.copyfile)
    print("lab files copied from your Git folder:", found)
else:
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(REPO_ZIP, timeout=120).read())) as z:
        for name in z.namelist():
            rel = name.split("/", 1)[1] if "/" in name else ""
            if rel.startswith(("src/", "data/")) and not name.endswith("/"):
                (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
                (ROOT / rel).write_bytes(z.read(name))
    print("lab files downloaded from GitHub")
for folder in ("out", "data/cache", "data/logs"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

## Step 1. Swap in the real models, and check they answer you

**What you will do.** Run the next cell. It sets up the three swaps and nothing else:

- `DatabricksEmbedder` sends text to the embedding model, four pieces at a time, and scales each
  result to length 1. Meaning search then compares question and chunk exactly as it did in Lab 3.
- `llm_rerank` sends the question and the top 15 candidates to the chat model, which grades each one
  from 0 (unrelated) to 3 (directly answers). The candidates are then sorted by grade. This is the
  same prompt the Azure track uses.
- `read_scan` sends the scanned escalation guide to `ai_parse_document`, because that file has no text
  layer for the normal PDF reader to find.

Before anything long starts, the cell sends one tiny request to each model.

**What you should see.** `models: databricks-gte-large-en and databricks-meta-llama-3-3-70b-instruct`,
then `both models answered: this run is live`. If you see "The models refused this run" instead, the rest of the notebook still runs and shows the recorded result.

**Why this step is here.** Only the parts that were stand-ins change. The search code, the merge by
rank, the recency filter and the scoring are the ones you ran in Lab 3, so any change in the numbers
comes from the models, not from different code.

In [ ]:
import re, json, time, threading
import numpy as np
from databricks.sdk import WorkspaceClient

EMBED_MODEL = "databricks-gte-large-en"
RERANK_MODEL = "databricks-meta-llama-3-3-70b-instruct"
_ws = WorkspaceClient()   # signs in as whoever runs the notebook: no key to create or store
def ask(model, body):
    """One call to a model this workspace serves, through Databricks' own client. The
    built-in models allow only a few requests a second per workspace, so a refused call
    waits and tries again."""
    for attempt in range(8):
        try:
            return _ws.api_client.do("POST", f"/serving-endpoints/{model}/invocations", body=body)
        except Exception as e:
            if "REQUEST_LIMIT" not in str(e) and "429" not in str(e) or attempt == 7:
                raise
            time.sleep(1.5 * (attempt + 1))
USE = {"embed_calls": 0, "chat_calls": 0, "tokens_in": 0, "tokens_out": 0}
_lock = threading.Lock()

class DatabricksEmbedder:
    """Meaning search with a served embedding model. Vectors are scaled to length 1, so the
    index's dot product is the same cosine similarity the class notebook uses."""
    name = EMBED_MODEL
    def __init__(self):
        self.memo = {}
    def fit(self, texts):
        return self
    def encode(self, texts):
        todo = [t for t in dict.fromkeys(texts) if t not in self.memo]
        for i in range(0, len(todo), 4):
            batch = todo[i:i + 4]
            r = ask(EMBED_MODEL, {"input": batch})
            with _lock:
                USE["embed_calls"] += 1
            for t, d in zip(batch, sorted(r["data"], key=lambda d: d["index"])):
                self.memo[t] = d["embedding"]
        V = np.array([self.memo[t] for t in texts], dtype="float32")
        return V / np.linalg.norm(V, axis=1, keepdims=True)

def llm_rerank(query, candidates, k=10, max_chars=600):
    """The Azure track's reranker prompt, sent to a served chat model. It grades each passage
    against the question from 0 to 3, then sorts. Ties keep the search order."""
    if not candidates:
        return []
    passages = "\n\n".join(f"[{i}] {' '.join(c.text.split())[:max_chars]}"
                            for i, (_, c) in enumerate(candidates))
    msgs = [
        {"role": "system", "content":
            "You grade how well each passage answers the query. Return only JSON: "
            '{"scores": [{"i": <passage number>, "score": <0 to 3>}]}. '
            "3 = directly answers, 2 = relevant, 1 = related topic, 0 = unrelated. "
            "Grade every passage."},
        {"role": "user", "content": f"Query: {query}\n\nPassages:\n{passages}"},
    ]
    scores = {}
    for attempt in range(3):
        try:
            r = ask(RERANK_MODEL, {"messages": msgs, "temperature": 0,
                                   "max_tokens": 60 + 18 * len(candidates)})
            with _lock:
                USE["chat_calls"] += 1
                USE["tokens_in"] += (r.get("usage") or {}).get("prompt_tokens", 0) or 0
                USE["tokens_out"] += (r.get("usage") or {}).get("completion_tokens", 0) or 0
            text = r["choices"][0]["message"]["content"] or ""
            found = re.search(r"\{.*\}", text, re.S)
            scores = {int(s["i"]): float(s["score"]) for s in json.loads(found.group(0))["scores"]}
            break
        except Exception as e:
            if attempt == 2:
                print("reranker gave no usable grades for:", query[:50], "|", type(e).__name__)
            time.sleep(2 * (attempt + 1))
    ranked = sorted(range(len(candidates)), key=lambda i: (-scores.get(i, 0.0), i))
    return [(scores.get(i, 0.0) / 3.0, candidates[i][1]) for i in ranked[:k]]

from layout_tools import ai_parse_read
def read_scan(path):
    """Only called for a PDF that gave no text at all: the scanned escalation guide."""
    try:
        parsed, seconds = ai_parse_read([path])
        text = parsed[path.name]["text"]
    except Exception as e:
        print("ai_parse_document is not available to you, so the scanned PDF stays unread:", type(e).__name__)
        return ""
    print(f"ai_parse_document read {path.name}: {len(text)} characters in {seconds:.0f}s")
    return text
print("models:", EMBED_MODEL, "and", RERANK_MODEL)
LIVE = True
try:
    ask(EMBED_MODEL, {"input": ["refund policy"]})
    ask(RERANK_MODEL, {"messages": [{"role": "user", "content": "Reply with the word ok."}], "max_tokens": 5})
    print("both models answered: this run is live")
except Exception as e:
    LIVE = False
    print("The models refused this run (" + type(e).__name__ + "). Most likely too many people are running")
    print("it at once, or your account cannot use them. The rest of the notebook shows the recorded result.")

## Step 2. Build the corpus with the real models

**What you will do.** Run the next cell. It loads the same seven files as Lab 1, cuts them into
chunks the same way, and turns every chunk into numbers with the real embedding model.

**What you should see.** A line saying `ai_parse_document read support_escalation_sop_scanned.pdf`,
with about 800 characters, then `31 pieces of text (Lab 3 has 30)`. Expect 15 to 30
seconds in total.

**Why this step is here.** The one extra chunk is the scanned escalation guide. In Lab 1 and Lab 3
the normal PDF reader got nothing from it, so it was never indexed. Remember question R17, which scored
0.000 at every stage of Lab 3: its answer is in that file. Watch what happens to it in Step 4.

In [ ]:
if LIVE:
    from freshcart import load_all, chunk_docs, apply_ocr, VectorIndex, current_only
    from labkit import Corpus, load_golden, relevance, table
    from retrieval import Hybrid, ndcg_at_k

    docs = apply_ocr(load_all(RAW), read_scan, RAW)
    chunks = chunk_docs(docs)
    t0 = time.perf_counter()
    index = VectorIndex(DatabricksEmbedder()).build(chunks)
    corpus = Corpus(docs, chunks, index)
    hyb = Hybrid(index)
    print(f"{len(chunks)} pieces of text (Lab 3 has 30), embedded in {time.perf_counter() - t0:.1f}s")

## Step 3. Predict, then run the four stages

**What you will do.** First write two predictions in the `YOUR CALL` cell, using your Lab 3 numbers
(0.803, 0.806, 0.815 and 0.874) as the starting point. Then run the cell after it. It puts all 20
questions through the four stages, one question at a time:

1. meaning search only
2. plus keyword search, merged by rank
3. plus the reranker over the top 15
4. plus the recency filter, which drops chunks from withdrawn documents before ranking

Every stage is scored with nDCG@10, a score from 0 to 1 that reaches 1.0 when the best answers sit at
the top of the first 10 results, and timed per question.

**What you should see.** A table of 20 rows, one per question, with four scores each, after about
two to four minutes. Most rows reach 1.000 by the last column.

**Why this step is here.** One change at a time, on the same questions, is the only way to say which
change earned its place. That rule does not change when the models do.

In [ ]:
# YOUR CALL: the average nDCG@10 after all four stages with real models, 0 to 1
PREDICT_FINAL = 0.90
# YOUR CALL: which stage adds the most with real models? "keyword", "rerank" or "recency"
PREDICT_BIGGEST = "recency"

In [ ]:
if LIVE:
    import statistics
    from concurrent.futures import ThreadPoolExecutor
    K, K_EACH, RERANK_TOP = 10, 50, 15
    golden = load_golden("retrieval_golden.json")["queries"]
    STAGES = ["dense", "hybrid", "rerank", "filter"]

    def run_stages(q):
        # Every stage starts by turning the question into numbers. That step is timed once and
        # added to each stage, so each time is what a user would wait for that stage alone.
        t = {}
        s = time.perf_counter(); index.embedder.encode([q]); embed = time.perf_counter() - s
        s = time.perf_counter(); dense = index.search(q, k=K_EACH); t["dense"] = embed + time.perf_counter() - s
        s = time.perf_counter(); fused = hyb.search(q, k=K_EACH, k_each=K_EACH); t["hybrid"] = embed + time.perf_counter() - s
        s = time.perf_counter(); rr = llm_rerank(q, fused[:RERANK_TOP], k=RERANK_TOP) + fused[RERANK_TOP:]
        t["rerank"] = t["hybrid"] + time.perf_counter() - s
        s = time.perf_counter()
        cur = hyb.search(q, k=K_EACH, k_each=K_EACH, where=current_only)
        filt = llm_rerank(q, cur[:RERANK_TOP], k=RERANK_TOP) + cur[RERANK_TOP:]
        t["filter"] = embed + time.perf_counter() - s
        return {"dense": dense, "hybrid": fused, "rerank": rr, "filter": filt}, t

    t_all = time.perf_counter()
    with ThreadPoolExecutor(1) as pool:
        runs = list(pool.map(lambda it: run_stages(it["query"]), golden))
    print(f"20 questions x 4 stages in {time.perf_counter() - t_all:.0f}s")

    rows, lat = [], {s: [] for s in STAGES}
    for item, (res, t) in zip(golden, runs):
        rel = relevance(item, corpus.chunks)
        row = {"id": item["id"], "query": item["query"][:40]}
        for s in STAGES:
            ids = [c.id for _, c in res[s]]
            row[s] = ndcg_at_k(ids, rel, k=K)
            lat[s].append(t[s] * 1000)
        row["week 1"] = "yes" if item.get("week1_failure") else ""
        rows.append(row)
    print(table(rows, ["id", "query", "dense", "hybrid", "rerank", "filter", "week 1"]))

## Step 4. Compare with Lab 3

**What you will do.** Run the next cell. It puts your Lab 3 averages next to this run's, with the
time per question, then lists the Week 1 failures, any question that got worse after reranking, and
R17. Last, it checks your two predictions.

**What you should see.** Close to these, measured on 8 October:

| Stage | Lab 3 | Real models | Time per question (median) |
|---|---|---|---|
| meaning only | 0.803 | 0.862 | about 0.24 s |
| + keyword | 0.806 | 0.908 | about 0.24 s |
| + rerank | 0.815 | 0.953 | about 3 s |
| + recency | 0.874 | 0.987 | about 3 s |

Your numbers can differ a little from a classmate's, because the reranker is a language model and can
grade the same passage differently from run to run. Times also depend on how busy the workspace is.

**Why this step is here.** With real models, every stage adds something, in roughly equal steps.
In Lab 3 the recency filter did most of the work. Both results are true: what a stage is worth
depends on the models behind it, which is why you measure instead of trusting the standard advice.

In [ ]:
import statistics
from labkit import table
CLASS = {"dense": 0.803, "hybrid": 0.806, "rerank": 0.815, "filter": 0.874}
NAMES = {"dense": "1 meaning only", "hybrid": "2 + keyword", "rerank": "3 + rerank", "filter": "4 + recency"}
# The recorded run of 8 Oct 2026, shown when the models refuse this one.
RECORDED = {"dense": (0.862, 233, 518), "hybrid": (0.908, 233, 517),
            "rerank": (0.953, 3723, 6796), "filter": (0.987, 3524, 5311)}
summary, means = [], {}
for s in ["dense", "hybrid", "rerank", "filter"]:
    if LIVE:
        ms = sorted(lat[s])
        real, p50, p95 = round(statistics.mean(r[s] for r in rows), 3), round(statistics.median(ms)), round(ms[int(0.95 * len(ms)) - 1])
    else:
        real, p50, p95 = RECORDED[s]
    means[s] = real
    summary.append({"stage": NAMES[s], "Lab 3": CLASS[s], "real models": real, "p50 ms": p50, "p95 ms": p95})
print(("THIS RUN" if LIVE else "RECORDED RUN (8 Oct)") + ", mean nDCG@10 over 20 questions\n")
print(table(summary, ["stage", "Lab 3", "real models", "p50 ms", "p95 ms"]))

if LIVE:
    print("\nWeek 1 failures, first stage to last:")
    for r in rows:
        if r["week 1"]:
            verdict = "still failing" if r["filter"] < 0.5 else "improved" if r["filter"] - r["dense"] >= 0.1 else "no change"
            print(f"  {r['id']}  {r['query']:40s} {r['dense']:.2f} -> {r['filter']:.2f}  {verdict}")
    worse = [r for r in rows if r["rerank"] < r["hybrid"] - 1e-9]
    print("\nQuestions that got WORSE after reranking:", ", ".join(f"{r['id']} {r['hybrid']:.2f} -> {r['rerank']:.2f}" for r in worse) or "none this run")
    r17 = next(r for r in rows if r["id"] == "R17")
    print(f"R17, whose answer is only in the scanned PDF: Lab 3 0.000, here {r17['filter']:.3f}")
    print(f"\nModel use: {USE['embed_calls']} embedding calls, {USE['chat_calls']} reranker calls")
else:
    print("\nIn the recorded run: R01 0.64 -> 0.97, R02 0.88 -> 0.98, R10 1.00 throughout, R17 0.000 -> 1.000,")
    print("and no question got worse after reranking.")

gains = {"keyword": means["hybrid"] - means["dense"], "rerank": means["rerank"] - means["hybrid"],
         "recency": means["filter"] - means["rerank"]}
biggest = max(gains, key=gains.get)
print(f"\nYour prediction for the final average: {PREDICT_FINAL:.2f}. Measured: {means['filter']:.3f}.")
print("Gain per stage: " + ", ".join(f"{k} {v:+.3f}" for k, v in gains.items()))
print(f"You predicted the biggest gain from {PREDICT_BIGGEST!r}; it came from {biggest!r}.")

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it.

---

**Pair:**

### 1. The two runs side by side

| Stage | Lab 3 | Real models | Median time |
|---|---|---|---|
| meaning search only | | | |
| + keyword search | | | |
| + reranking | | | |
| + recency filter | | | |

### 2. Which stage added the most with real models, and is it the same stage as in Lab 3?

### 3. The reranker now helps. Look at its median time. Would you ship it for a support agent on a live chat? Why?

### 4. R17 went from 0.000 to 1.000 without any change to search. What fixed it, and what does that say about where to spend effort first?